In [1]:
import os
import json
import re

In [26]:
PATH_AUDITS = './../datasets/production/audits.json'
PATH_ENTITIES = './../datasets/niort/entities.json'
PATH_USERS = './../datasets/production/users.json'

In [ ]:
# Load the PATH_AUDITS json into dict
entities = {}

with open(PATH_ENTITIES, 'r') as file:
    entities = json.load(file)

print(f'Loaded {len(entities)} entities from {PATH_ENTITIES}')


Loaded 6793 entities from ./../datasets/niort/entities.json


In [8]:
# Load the PATH_AUDITS json into dict
audits = {}
with open(PATH_AUDITS, 'r') as file:
    audits = json.load(file)

print(f'Loaded {len(audits)} audits from {PATH_AUDITS}')

Loaded 132605 audits from ./../datasets/production/audits.json


In [20]:
# aggregate audits by entity id
audits_by_entity = {}
for ai, audit in enumerate(audits):
    entity_id = audit.get('entityId')

    if entity_id:
        if entity_id not in audits_by_entity:
            audits_by_entity[entity_id] = []
        audits_by_entity[entity_id].append(audit)
print(f'Aggregated audits by entity, found {len(audits_by_entity)} entities with audits')

Aggregated audits by entity, found 40485 entities with audits


In [23]:
# iterate over entities, find all items in audits for the entityId and fill the  audits_by_user dictionary so the first key level is user, second key level is entityId, for each entityId on second level store the relevant audit item count, on top level, user should store the total count of audits

audits_by_user = {}
for entity_id, entity_audits in audits_by_entity.items():
    for audit in entity_audits:
        user_id = audit.get('user')
        if user_id:
            if user_id not in audits_by_user:
                audits_by_user[user_id] = {}
            audits_by_user[user_id][entity_id] = audits_by_user[user_id].get(entity_id, 0) + 1
        audits_by_user[user_id]['total'] = audits_by_user[user_id].get('total', 0) + 1
print(f'Aggregated audits by user, found {len(audits_by_user)} users with audits')

Aggregated audits by user, found 12 users with audits


In [36]:
# load the PATH_USERS json into dict where userId is the key

users = {}
with open(PATH_USERS, 'r') as file:
    users = json.load(file)

# Transform users list into a dictionary with userId as the key
users = {user['id']: user for user in users}
print(f'Loaded {len(users)} users from {PATH_USERS}')

Loaded 19 users from ./../datasets/production/users.json


In [37]:
users

{'0': {'active': False,
  'bookmarks': None,
  'email': 'import@inkvisitor.com',
  'id': '0',
  'name': 'import',
  'options': {'defaultLanguage': '',
   'defaultTerritory': '',
   'searchLanguages': []},
  'password': '$2b$10$Qd92YDSwwZJiUpZ/vapHbe/OVidQvdoA3lcBoDEEGjDVe9HjbHwV.',
  'rights': [],
  'role': 'admin',
  'verified': True},
 '02868c93-517e-4602-94b7-057ce1a393fd': {'active': True,
  'bookmarks': [{'entityIds': ['5fb52fa6-a585-41d4-be0c-84c40b6bf07a',
     '19ff6fa7-9f20-434b-a4a5-59a549fdbefd',
     '82f65869-6bf3-41a2-beb3-94cf798e6c05',
     'e9925b93-970e-4357-b336-d6b2ee7b9357',
     'd04adbe2-20a5-4958-9e68-e7de17462949',
     '3438609f-b809-4be1-888c-a53df7bb7c1c',
     '40cbd809-fa9a-4a9d-85fb-e72bed138312',
     '01670d02-19f7-40b3-a8ea-355ac0dd30a0',
     'a3e7db62-cd55-4438-ad03-98e33da4a632',
     '297b9aa7-39fa-4d3c-a64e-f087bc995259',
     'ceeeb03d-47c1-4b4d-a1bf-17ee5dd23f30',
     '4ce5e669-d421-40c9-b1ce-f476fdd171fe',
     '851ecfbd-1255-4c89-801f-00e819b

In [ ]:
# For each user in audits_by_user, find the user in users and add the user data to the audits_by_user dictionary
for user_id, user_audits in audits_by_user.items():
    user = users[user_id]
    if user:
        audits_by_user[user_id]['user'] = {
            'id': user_id,
            'name': user.get('name'),
            'email': user.get('email'),
            'role': user.get('role')
        }
    else:
        audits_by_user[user_id]['user'] = {
            'id': user_id,
            'name': None,
            'email': None,
            'role': None
        } 
print(f'Added user data to audits_by_user, now contains {len(audits_by_user)} users with audits')

101
02868c93-517e-4602-94b7-057ce1a393fd
103
100
102
1
5a8b6aff-3dc1-40b2-9704-82b47a952d4c
104
106
105
107
151
Added user data to audits_by_user, now contains 12 users with audits


In [48]:
# calculate percentage for each user
total_audits = sum(user_data.get('total', 0) for user_data in audits_by_user.values())
for user_id, user_data in audits_by_user.items():
    user_data['percentage'] = (user_data.get('total', 0) / total_audits * 100) if total_audits > 0 else 0

for user in audits_by_user:
  print(f'User {user} {audits_by_user[user]["user"]["name"]} has {audits_by_user[user].get("total", 0)} audits ({audits_by_user[user]["percentage"]:.2f}%)')

User 101 Robert Shaw has 32339 audits (24.39%)
User 02868c93-517e-4602-94b7-057ce1a393fd Katalin Suba has 22647 audits (17.08%)
User 103 Katia Riccardo has 38357 audits (28.93%)
User 100 David Zbíral has 22138 audits (16.69%)
User 102 Davor Salihovic has 9988 audits (7.53%)
User 1 admin has 2700 audits (2.04%)
User 5a8b6aff-3dc1-40b2-9704-82b47a952d4c stanislaw.banach has 3607 audits (2.72%)
User 104 Jan Král has 300 audits (0.23%)
User 106 Lidia Hinz-Wieczorek has 188 audits (0.14%)
User 105 Reima Välimäki has 108 audits (0.08%)
User 107 Larissa de Freitas Lyth has 224 audits (0.17%)
User 151 Tomáš Hampejs has 9 audits (0.01%)
